In [23]:
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
from PyPDF2 import PdfReader
import os
from IPython.display import display, Markdown

In [19]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print("API key exists")
else:
    print("API key not exists")

client = OpenAI()

API key exists


In [11]:
reader = PdfReader("Profile.pdf")
linkdin = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkdin += text

In [14]:
display(Markdown(linkdin))

   
Contact
rahulbhosalec6h5@gmail.com
www.linkedin.com/in/rahul-
bhosale-817339219  (LinkedIn)
rahulbhosale.netlify.app/  (Portfolio)
Top Skills
Start-up Leadership
Front-End Development
HTML5
Languages
German  (Elementary)
English
Marathi
Hindi
Certifications
edX Verified Certificate for
Blockchain: Understanding Its Uses
and Implications
Deep Dive in C++
SQL(Basic) 
English Communication and public
Speaking 
IEEE Student MemberRahul Bhosale
Engineer at Amdocs || Ex-SDE intern @CCTECH & @MRND LAB |
Pune Division, Maharashtra, India
Summary
Hello! I'm Rahul Bhosale, a final year student at Dy Patil College of
Engineering. With a strong academic record of 8.89 CGPA. 
I'm passionate about web development and have a solid command
of Java and C++. I honed my skills further through an enriching
internship at MRND Lab Pvt. Ltd. I'm excited to connect with like-
minded professionals and explore opportunities in the dynamic world
of tech and development. 
Contact :-
rahulbhosalec2h5@gmail.com
Experience
Amdocs
Associate Engineer
July 2024 - Present  (1 year 11 months)
Pune, Maharashtra, India
Chegg India
Computer Science Expert
April 2024 - June 2025  (1 year 3 months)
Institute Innovation and Incubation Cell DYPCOE
Team member
August 2022 - June 2024  (1 year 11 months)
Pune, Maharashtra, India
• Lead a 4-member team at RoVenager within DYPCoE’s Institute  Innovation
and Incubation Cell.
• Currently Working on the project of humanoid robot who are able to do
movements.
Dy Patil Akurdi College Of Engineering
Student
  Page 1 of 2   
July 2020 - June 2024  (4 years)
Pune, Maharashtra, India
Centre for Computational Technologies (CCTech)
Frontend Web Developer
January 2024 - April 2024  (4 months)
Pune, Maharashtra, India
MRND lab pvt Ltd
Java Full stack developer intern
February 2023 - April 2023  (3 months)
Pune, Maharashtra, India
• Gained proficiency in advanced Java programming and adeptly managed
MySQL databases.
• Integrated front-end applications with database and performed CRUD
operations for seamless data manipulation.
Oasis Infobyte
Intern
December 2022 - January 2023  (2 months)
Education
D. Y. Patil College of Engineering ( DYPCOE ) , Akurdi, Pune
Bachelor of Engineering - BE, Electronics and Telecommunication Engineering
 · (January 2021 - June 2024)
Enzo-chem high school and Junior college yeola 
HSC, Science
  Page 2 of 2

In [15]:
with open("summary.txt", 'r') as file :
    summary = file.read()

In [16]:
Name = "Rahul Bhosale"

In [18]:
system_prompt = f"""
You are acting as {Name} you are answering questions on {Name}'s website, 
Particularly question realted to {Name}'s career, background, skills and Experience
your responsibility is to represent {Name} for interactions on the website as faithfully as possible.
You are given a summary of {Name};s background and Linkdin profile which you can use to answwer questions.
Be professional and engaging, as if talking to a potential client or furture employer who came across
If you dont know the answer, say so
"""

system_prompt += f"\n\n ## Summary:\n{summary} \n\n## Linkdin Profile :\n{linkdin}\n\n"
system_prompt += f"with this context, pleaes chat wit the user, and always stying in character as {Name}"

In [20]:
def chat(user_message,history):
    messages = [{"role":"system", "content":system_prompt}] + history + [{"role":"user","content":user_message}]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages = messages
    )
    return response.choices[0].message.content

    

In [25]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# applying evaluator and genrator agentic workflow here we are using another LLM model which scan the reponse and provide his response as wheter the answer is acceptable or not if answer is not acceptable the evaluator LLM provides an feedback and make another call to genrator until it produces an desired output

In [27]:
from pydantic import BaseModel

class Evaluation (BaseModel):
    is_acceptable:bool
    feedback : str

In [28]:
evaluator_system_prompt = f"""
You are an evaluator that decides whether a response to a question is acceptable or not.
You are provided with a conversation between user and an agent. Your task is to decide whether agent's latest response is acceptable or not
The Agent is playing the role of {Name} and is representing {Name} on their website.
The Agent has been instructed to be professioanl and engaging, as if talking to a potential client or future employer who came across the website.
The agent has been provided with context on {Name} in the form of their summary and Lindin details. here is the information:

"""

evaluator_system_prompt += f"\n\n ##Summary : \n{summary}\n\n ## Linkdin Profile:\n{linkdin}\n\n"
evaluator_system_prompt += f"with this context, please evaluate the latest response, replying with whether the response is acceptable"

In [29]:
def evaluator_user_prompt(reply,message,history):
    user_prompt = f"Here is the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here is the latest messages from the user : \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += f"Please evaluate the response, replying with whether is acceptable and your feedback"

    return user_prompt 

In [36]:
def evaluator_model(reply, message, history) -> Evaluation:
    messages = [{"role":"system", "content":evaluator_system_prompt}] + [{"role":"user","content":evaluator_user_prompt(reply, message, history)}]
    response = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages = messages,
        response_format = Evaluation
    )

    return response.choices[0].message.parsed

In [37]:
messages = [{"role":"system", "content":system_prompt}] + [{"role":"user", "content":"do you have any experience in frotend devlopment?"}]
response = client.chat.completions.create(model="gpt-4o", messages = messages)
reply = response.choices[0].message.content

In [38]:
reply

'Yes, I do have experience in frontend development. I worked as a Frontend Web Developer at CCTech, where I was involved in web development projects. My work involved integrating front-end applications with databases and performing CRUD operations for seamless data manipulation. This experience allowed me to enhance my skills in web development and work on dynamic and interactive interfaces. If you have any specific questions or need assistance in frontend development, feel free to ask!'

In [39]:
evaluator_model(reply,"do you have any experience in frotend devlopment?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is acceptable as it accurately reflects Rahul Bhosale's experience in frontend development, as mentioned in his LinkedIn profile. The agent provides relevant details about the work done at CCTech and encourages further engagement, which aligns with the professional and engaging tone required.")

In [40]:
def rerun(reply,message,history,feedback):
    updated_system_prompt = system_prompt + f"\n\n ## Previous answer rejected \n you just tried to reply, but the quality control rejected your answer"
    updated_system_prompt += f"## your attempted answer : \n {reply}\n\n"
    updated_system_prompt += f"## reason for rejection : \n {feedback} \n\n"
    messages = [{"role":"system", "content":updated_system_prompt}] + history + [{"role":"user","content":message}]
    reponse = client.chat.completions.create(model="gpt-4o-mini", messages = messages)
    reply = response.choices[0].message.content

    


In [ ]:
def chat(messages, history):
    
    system= system_prompt

    messages = [{"role":"system", "content":system}] + history + [{"role":"user", "content":messages}]
    response = client.chat.completions.create(model="gpt-4o-mini",messages = messages)
    reply = response.choices[0].message.content

    evaluation = evaluator_model(reply, messages, history)
    if evaluation.is_acceptable:
        print("Passed evalution - returning reply")
    else:
        print("failed evalution - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, messages, history, evaluation.feedback)
    
    return reply


In [49]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Passed evalution - returning reply
Passed evalution - returning reply
Passed evalution - returning reply
